# AI-Driven Adaptive Smart Agriculture – Machine Learning Component
**How to use this notebook (Google Colab or local Jupyter)**
1. Put these files in one folder: `agri_features.py`, `irrigation_core.py`, this notebook and the workbook
   `master_smart_agriculture_dataset.xlsx`.
2. In Colab: upload the folder to Google Drive, mount Drive (cell below) and `%cd` into the folder.
3. Run the cells from top to bottom. Every section prints what it found, so read the output.

**Honesty rules built into this notebook**
* The original workbook is never modified (its SHA-256 is printed so you can prove it).
* A model is only called "useful" if it beats the *persistence baseline* ("moisture stays the same") on a
  chronological test set, with a confidence interval. Otherwise it is reported as *not useful*.
* Crop moisture thresholds are **not invented**. They stay empty (`null`) until you fill them from a cited
  agronomic reference and your own sensor calibration.


In [ ]:
# 0. Setup
import os, sys, json, glob, hashlib, warnings, datetime as dt
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:                                    # uncomment if your files are on Drive
    # from google.colab import drive; drive.mount('/content/drive'); %cd /content/drive/MyDrive/smart_agri
    pass
sys.path.insert(0, os.getcwd())
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score,
                             recall_score, f1_score, balanced_accuracy_score, confusion_matrix)
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False
import agri_features as AF

OUT = "outputs"; os.makedirs(f"{OUT}/audit", exist_ok=True); os.makedirs(f"{OUT}/plots", exist_ok=True)
os.makedirs("models", exist_ok=True)
SEED = 42
def show(fig_name):
    plt.tight_layout(); plt.savefig(f"{OUT}/plots/{fig_name}.png", dpi=110)
    plt.show() if IN_COLAB else plt.close()

## 1. Load the Excel workbook (read-only)
Each worksheet is read separately. Nothing is concatenated across sheets because they describe different things.


In [ ]:
# 1. Load
WORKBOOK = "master_smart_agriculture_dataset.xlsx"
if not os.path.exists(WORKBOOK):
    cand = glob.glob("**/*master_smart_agriculture_dataset.xlsx", recursive=True) + glob.glob("/mnt/user-data/attachments/*/*master_smart_agriculture_dataset.xlsx")
    if cand: WORKBOOK = cand[0]
    elif IN_COLAB:
        from google.colab import files; up = files.upload(); WORKBOOK = list(up)[0]
    else: raise FileNotFoundError("Place master_smart_agriculture_dataset.xlsx next to this notebook")
sha = hashlib.sha256(open(WORKBOOK, "rb").read()).hexdigest()
print("Workbook:", WORKBOOK, "\nSHA-256 :", sha)
RAW_SHEETS = pd.read_excel(WORKBOOK, sheet_name=None, header=None, dtype=object)

def as_table(name):
    """Sheet -> DataFrame using the first row as header. Returns a COPY; RAW_SHEETS stays untouched."""
    d = RAW_SHEETS[name].copy()
    d.columns = [str(c).strip() for c in d.iloc[0]]
    return d.iloc[1:].reset_index(drop=True)

def smart_numeric(df, min_share=0.95):
    """Convert a column to numbers only if >=95% of its non-empty cells are numeric (so text stays text)."""
    out = df.copy()
    for c in out.columns:
        conv = pd.to_numeric(out[c], errors="coerce")
        nn = out[c].notna().sum()
        if nn and conv.notna().sum() / nn >= min_share:
            out[c] = conv
    return out

## 2. Inspect every worksheet
Records, columns, data types, missing values, duplicates, and (for the sensor sheet) timestamps.


In [ ]:
# 2. Inspect
DOC_SHEETS = ["START_HERE", "DATA_DICTIONARY", "EcoIrrigate_Quality", "ESP32_Field_Data_Template"]
summary = []
for name in RAW_SHEETS:
    t = smart_numeric(as_table(name))
    ncols = t.shape[1]
    colrep = pd.DataFrame({"dtype": t.dtypes.astype(str), "missing": t.isna().sum(),
                           "missing_pct": (t.isna().mean() * 100).round(2), "n_unique": t.nunique()})
    num = t.select_dtypes("number")
    colrep["min"], colrep["max"] = num.min(), num.max()
    colrep.to_csv(f"{OUT}/audit/{name}_columns.csv")
    summary.append(dict(sheet=name, records=len(t), columns=ncols,
                        missing_cells=int(t.isna().sum().sum()),
                        exact_duplicate_rows=int(t.astype(str).duplicated().sum()) if len(t) else 0))
audit = pd.DataFrame(summary); audit.to_csv(f"{OUT}/audit/sheet_summary.csv", index=False)
print(audit.to_string(index=False))

# Sensor-sheet timestamp / sampling check (we do NOT assume 15 minutes - we measure it)
_s = smart_numeric(as_table("EcoIrrigate_Sensor_Data")); _s["Timestamp"] = pd.to_datetime(_s["Timestamp"])
for farm, g in _s.groupby("Farm_ID"):
    dts = g.sort_values("Timestamp").Timestamp.diff().dropna().value_counts()
    print(f"{farm}: {g.Timestamp.min()} -> {g.Timestamp.max()}  rows={len(g)}  intervals={dts.to_dict()}")
print("\nWorksheets that are documentation/templates, not training data:", DOC_SHEETS)
print("ESP32 template columns:", len(as_table("ESP32_Field_Data_Template").columns), "| rows of real data:", len(as_table("ESP32_Field_Data_Template")))

### Which dataset can support which ML task? (decided from the audit above)
| Task | Dataset that can be used | Verdict |
|---|---|---|
| A. Soil-moisture analysis (validity + relative state) | `EcoIrrigate_Sensor_Data` | Possible – but only *relative* (own-history) states, no agronomic dry/wet labels |
| B. Moisture forecasting 1 h / 6 h / 24 h | `EcoIrrigate_Sensor_Data` (2 Kolkata sensors, Jan–Apr 2025) | Possible as an experiment. No crop, rain, light or humidity columns → **not crop-aware** |
| C. Moisture-reduction (drying) analysis | `EcoIrrigate_Sensor_Data` | Possible, timestamp-aware |
| D. Irrigation decision | none has irrigation labels | **Rule-based + safety logic**; ML only advises. Needs your own logged outcomes |
| E. Water requirement | none has pump / flow data | **Not trainable**. Needs ESP32 + flow sensor logs |
| Crop envelopes (config) | `Crop_Recommendation_Data`, `Benue_Crop_Data`, `India_District_Crop_Context`, `FAO_India_Calendar` | Climate/season context only. **No moisture targets** |
| Per-crop moisture model | none (sensor sheet has no crop column) | **Unsupported** |


## 3. Clean data (originals preserved)
* Sensor sheet: parse timestamps, sort per farm, drop exact duplicate (farm, timestamp), flag (don't delete) outliers,
  convert pressure inHg → hPa. Wetting events are **real** physical events, so they are kept.
* Benue sheet: 4 000 of 5 000 rows are exact duplicates → keep unique rows. Strip trailing spaces in labels.
* Crop names: map synonyms **only** where we are sure (rice/Rice, tomato/tomatoes...). Groups are not treated as crops.


In [ ]:
# 3. Clean
INCHG_TO_HPA = 33.8639
PHYSICAL_LIMITS = {"Volumetric_Moisture_Pct": (0, 100), "Atm_Temperature_C": (-10, 60), "Soil_Temperature_C": (-10, 60),
                   "Atm_Pressure_inHg": (25, 32), "Raw_Capacitive_ADC": (0, 1023)}   # 10-bit ADC in THIS dataset (max 833)

def clean_sensor(df):
    s = smart_numeric(df.copy()); s["Timestamp"] = pd.to_datetime(s["Timestamp"])
    n0 = len(s)
    s = s.drop_duplicates(["Farm_ID", "Timestamp"]).sort_values(["Farm_ID", "Timestamp"]).reset_index(drop=True)
    s["pressure_hpa"] = s["Atm_Pressure_inHg"] * INCHG_TO_HPA
    bad = pd.Series(False, index=s.index)
    for c, (lo, hi) in PHYSICAL_LIMITS.items():
        bad |= ~s[c].between(lo, hi)
    s["physically_invalid"] = bad
    # robust outlier FLAG (median/MAD z-score over a 24 h rolling window) - flagged, never removed
    for c in ["Volumetric_Moisture_Pct", "Atm_Temperature_C", "Soil_Temperature_C"]:
        med = s.groupby("Farm_ID")[c].transform(lambda x: x.rolling(96, min_periods=24, center=True).median())
        mad = s.groupby("Farm_ID")[c].transform(lambda x: (x - x.rolling(96, min_periods=24, center=True).median()).abs().rolling(96, min_periods=24, center=True).median())
        s[f"{c}_outlier_flag"] = ((s[c] - med).abs() / (1.4826 * mad.replace(0, np.nan))) > 6
    print(f"sensor: {n0} -> {len(s)} rows | physically invalid: {int(bad.sum())} | "
          f"moisture outlier flags (kept): {int(s['Volumetric_Moisture_Pct_outlier_flag'].sum())}")
    return s

CROP_ALIASES = {  # canonical key -> names exactly as reported in the sheets. Groups (e.g. 'Vegetables') are NOT aliases.
    "rice": ["rice", "Rice", "Rice one", "Rice two"], "wheat": ["Wheat"], "maize": ["maize", "Maize"],
    "tomato": ["Tomato", "tomatoes"], "banana": ["banana", "Banana"], "cotton": ["cotton", "Cotton", "Cotton(lint)"],
    "sugarcane": ["Sugarcane"], "chilli": ["Dry chillies"], "potato": ["Potato"],
    "soybean": ["soybean", "Soyabean", "Soybeans"], "mango": ["mango", "Mango"], "orange": ["orange", "Orange"],
    "papaya": ["papaya", "Papaya"], "jute": ["jute", "Jute"], "groundnut": ["Groundnut", "Groundnuts"],
    "coffee": ["coffee"], "coconut": ["coconut"], "grapes": ["grapes"], "apple": ["apple"],
}
ALIAS_LOOKUP = {n.strip().lower(): k for k, v in CROP_ALIASES.items() for n in v}
def canon(name): 
    n = str(name).strip().lower(); return ALIAS_LOOKUP.get(n, n.replace(" ", "_"))

sensor = clean_sensor(as_table("EcoIrrigate_Sensor_Data"))
crop_rec = smart_numeric(as_table("Crop_Recommendation_Data")); crop_rec["crop_key"] = crop_rec["label"].map(canon)
benue_raw = smart_numeric(as_table("Benue_Crop_Data"))
benue = benue_raw.drop_duplicates().copy()
benue["soil"] = benue["soil"].str.strip(); benue["crop_key"] = benue["crop_label"].map(canon)
print(f"benue: {len(benue_raw)} -> {len(benue)} unique rows (removed {len(benue_raw)-len(benue)} exact duplicates)")
india = smart_numeric(as_table("India_District_Crop_Context")); india["crop_key"] = india["dominant_crop"].map(canon)

# FAO calendar: the sheet holds several stacked blocks, each starting at a repeated 'Irrigated crops' header row.
fao = as_table("FAO_India_Calendar"); fao.insert(0, "block", (fao["crop_or_group"] == "Irrigated crops").cumsum())
fao = fao[~fao["crop_or_group"].isin(["Irrigated crops"]) & ~fao["crop_or_group"].str.startswith("Area equipped")].copy()
MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fao = smart_numeric(fao)
print("FAO calendar blocks:", fao.groupby("block").size().to_dict(), "(block meaning is not documented in the workbook)")

## 4. Exploratory data analysis (what does the real moisture data look like?)


In [ ]:
# 4. EDA
fig, ax = plt.subplots(3, 1, figsize=(12, 9))
for farm, g in sensor.groupby("Farm_ID"):
    ax[0].plot(g.Timestamp, g.Volumetric_Moisture_Pct, lw=.7, label=farm)
ax[0].set_title("Volumetric moisture (%) – Kolkata, Jan–Apr 2025 (steps up = wetting events, cause unlabeled)"); ax[0].legend()
g = sensor
ax[1].scatter(g.Raw_Capacitive_ADC, g.Volumetric_Moisture_Pct, s=2, alpha=.3)
ax[1].set_xlabel("Raw capacitive ADC"); ax[1].set_ylabel("Moisture %"); ax[1].set_title(f"ADC vs moisture (corr {g.Raw_Capacitive_ADC.corr(g.Volumetric_Moisture_Pct):.3f}) – same sensor, so ADC is not an independent predictor")
sensor.assign(hour=sensor.Timestamp.dt.hour).groupby("hour").Volumetric_Moisture_Pct.mean().plot(ax=ax[2]); ax[2].set_title("Mean moisture by hour of day")
show("eda_moisture")
wet_events = sensor.groupby("Farm_ID").apply(lambda g: int((g.Volumetric_Moisture_Pct.diff() > AF.RISE_PP).sum()))
print("Wetting events (rise > 1 pp in one 15-min step) per farm:", wet_events.to_dict(),
      "\n-> only a handful, and the workbook does not say whether each was rain or irrigation.")
print("Columns present in sensor data that the plan needs but are ABSENT:",
      ["crop", "growth_stage", "air humidity (DHT11)", "light (LDR)", "rain", "pump/relay state", "water volume"])
print("Crop_Recommendation samples per crop:", crop_rec.crop_key.value_counts().head(5).to_dict(), "... (", crop_rec.crop_key.nunique(), "crops )")

## 5. Prepare the crop configuration
Everything that the workbook genuinely supports is filled in (names across sheets, climate envelopes, India context,
FAO calendar months). **Moisture targets, depletion fractions and water requirements are left `null`** because no sheet
contains verified values. The irrigation engine refuses to irrigate a crop whose targets are still `null`.


In [ ]:
# 5. Crop configuration
PRIORITY = ["rice", "wheat", "maize", "tomato", "banana", "cotton", "sugarcane", "chilli", "potato"]
# FAO calendar names (exact crop vs. only a crop GROUP – groups are flagged, never treated as the crop itself)
FAO_MAP = {"rice": (["Rice", "Rice one", "Rice two"], "exact"), "wheat": (["Wheat"], "exact"), "maize": (["Maize"], "exact"),
           "cotton": (["Cotton"], "exact"), "sugarcane": (["Sugarcane"], "exact"), "soybean": (["Soybeans"], "exact"),
           "groundnut": (["Groundnuts"], "exact"),
           "potato": (["Potatoes and sweet potatoes"], "group_only"), "tomato": (["Vegetables"], "group_only"),
           "chilli": (["Vegetables"], "group_only"), "banana": (["Fruits"], "group_only")}

def env(df, cols, key):
    g = df[df.crop_key == key]
    if g.empty: return None
    out = {"n_samples": int(len(g))}
    for c in cols:
        out[c] = {"p5": round(float(g[c].quantile(.05)), 2), "median": round(float(g[c].median()), 2), "p95": round(float(g[c].quantile(.95)), 2)}
    return out

def india_ctx(key):
    g = india[india.crop_key == key]
    if g.empty: return None
    return {"district_year_rows_where_dominant": int(len(g)), "top_seasons": g.season.value_counts().head(3).to_dict(),
            "top_states": g.state.value_counts().head(3).to_dict(),
            "median_temp_mean_c": round(float(g.temp_mean_c.median()), 1), "median_humidity_pct": round(float(g.humidity_pct.median()), 1),
            "median_annual_rainfall_mm": round(float(g.annual_rainfall_mm.median()), 0),
            "note": "district-level historical context, not field conditions"}

def fao_ctx(key):
    if key not in FAO_MAP: return None
    names, match = FAO_MAP[key]
    rows = []
    for _, r in fao[fao.crop_or_group.isin(names)].iterrows():
        rows.append({"block": int(r.block), "name_in_sheet": r.crop_or_group, "months_with_entry": [m for m in MONTHS if pd.notna(r[m])]})
    return {"match_type": match, "entries": rows, "caveat": "India-East 2006 calendar; block meaning not documented"} if rows else None

all_keys = sorted(set(PRIORITY) | set(crop_rec.crop_key) | set(benue.crop_key) | set(india.crop_key))
crop_config = {"_readme": "Fields set to null are NOT supported by the workbook. Fill them only from a cited reference (FAO-56 / ICAR / "
                          "state agri. university) plus calibration of YOUR sensor in YOUR soil, using irrigation_core.CropRegistry.set_moisture().",
               "_created": dt.datetime.now().isoformat(timespec="seconds"), "_workbook_sha256": sha,
               "sensor_calibration": {"adc_dry": None, "adc_wet": None, "method": "air-dry reading and saturated-soil reading of YOUR sensor, 12-bit ESP32 ADC"},
               "pump_calibration": {"flow_l_per_min": None, "flow_sensor_pulses_per_litre": None, "method": "bucket test / flow-meter calibration"},
               "crops": {}}
for k in all_keys:
    crop_config["crops"][k] = {
        "display_name": k.replace("_", " ").title(), "priority_crop": k in PRIORITY,
        "names_in_workbook": sorted({n for n, kk in ALIAS_LOOKUP.items() if kk == k} | ({k} if k in set(crop_rec.crop_key) | set(benue.crop_key) else set())),
        "data_support": {"in_crop_recommendation": bool((crop_rec.crop_key == k).any()), "in_benue": bool((benue.crop_key == k).any()),
                         "in_india_context_as_dominant": bool((india.crop_key == k).any()), "in_fao_calendar": k in FAO_MAP,
                         "in_sensor_moisture_data": False},
        "climate_envelope_crop_recommendation": env(crop_rec, ["temperature", "humidity", "rainfall", "ph"], k),
        "climate_envelope_benue_nigeria": env(benue, ["temperature", "humidity", "rainfall", "ph"], k),
        "india_district_context": india_ctx(k), "fao_calendar": fao_ctx(k),
        "moisture": {"min_pct": None, "target_pct": None, "max_pct": None, "unit": "calibrated volumetric % of YOUR sensor",
                     "source": None, "status": "REQUIRES_AGRONOMIC_REFERENCE_AND_LOCAL_CALIBRATION"},
        "stage_overrides": {}, "stage_names_hint": ["initial", "development", "mid", "late"],
        "management_allowed_depletion_p": None, "water_requirement": {"seasonal_mm": None, "source": None}}
for p in (OUT + "/crop_config.json", "crop_config.json"):
    json.dump(crop_config, open(p, "w"), indent=1, default=str)
cc = pd.DataFrame([{"crop": k, **{f"has_{kk[3:]}": v for kk, v in c["data_support"].items()}, "moisture_target_available": c["moisture"]["target_pct"] is not None}
                   for k, c in crop_config["crops"].items()])
print(len(cc), "crops in config.  Priority crops:\n", cc[cc.crop.isin(PRIORITY)].to_string(index=False))
print("\nCrops with a moisture target supported by the workbook:", int(cc.moisture_target_available.sum()), "(none – by design)")
print("Crops with ANY sensor-moisture training data:", int(cc.has_sensor_moisture_data.sum()))

## 6. Engineer time-series features
All features at time *t* use only past data; all targets look into the future. Shifts are done on a regular
15-minute grid so a missing interval becomes `NaN` instead of silently shifting the data.


In [ ]:
# 6. Features
STEP = AF.infer_step_minutes(sensor.Timestamp); print("Measured sampling interval:", STEP, "min")
ren = {"Volumetric_Moisture_Pct": AF.MOISTURE, "Atm_Temperature_C": AF.AIR_T, "Soil_Temperature_C": AF.SOIL_T, "pressure_hpa": AF.PRESS}
GRIDS = {farm: AF.to_regular_grid(g.rename(columns=ren), "Timestamp", STEP, list(ren.values())) for farm, g in sensor.groupby("Farm_ID")}
t0, t1 = min(g.index.min() for g in GRIDS.values()), max(g.index.max() for g in GRIDS.values())
TRAIN_END, VAL_END = t0 + (t1 - t0) * 0.70, t0 + (t1 - t0) * 0.85
print(f"chronological split  train: {t0.date()}..{TRAIN_END.date()} | validation: ..{VAL_END.date()} | test: ..{t1.date()}")
train_m = pd.concat([g.loc[:TRAIN_END, AF.MOISTURE] for g in GRIDS.values()])
DRY_THR = float(train_m.quantile(0.15))     # RELATIVE threshold, from TRAIN data only (demo of classification metrics)

def build(feature_set, hname):
    spp = 60 // STEP; h = AF.HORIZONS_H[hname]; n = h * spp; parts = []
    for farm, g in GRIDS.items():
        X = AF.make_features(g, STEP, feature_set); Y = AF.make_targets(g, STEP, DRY_THR)
        d = X.join(Y[[f"delta_{hname}", f"y_{hname}", f"event_in_{hname}", "dry_risk"]]).dropna()
        d["farm"] = farm
        d["split"] = np.select([d.index <= TRAIN_END - pd.Timedelta(hours=h), (d.index > TRAIN_END) & (d.index <= VAL_END - pd.Timedelta(hours=h)), d.index > VAL_END],
                               ["train", "val", "test"], default="purged")   # purge rows whose target looks into the next split
        parts.append(d)
    return pd.concat(parts)

## 7. Train models (tasks B, C, A)
We predict the **change** in moisture over the horizon (Δ) and add it back to the current value. Baselines:
*persistence* (Δ = 0) and *linear trend* (extrapolate the last hour).


In [ ]:
# 7. Train
def model_zoo():
    z = {"Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
         "RandomForest": RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.6, n_jobs=-1, random_state=SEED),
         "GradientBoosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=4, l2_regularization=1.0, random_state=SEED)}
    if HAS_XGB: z["XGBoost"] = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=SEED, verbosity=0)
    return z

def reg_metrics(y, p, cur):
    return dict(MAE=mean_absolute_error(y, p), RMSE=mean_squared_error(y, p) ** 0.5, R2=r2_score(y, p), R2_delta=r2_score(y - cur, p - cur), n=len(y))

FITTED, PRED, rows = {}, {}, []
for fs in ["deployable", "extended"]:
    for hname in AF.HORIZONS_H:
        D = build(fs, hname); cols = AF.FEATURE_SETS[fs]; tgt = f"delta_{hname}"
        for sp in ["train", "val", "test"]:
            S = D[D.split == sp]
            base = {"Persistence (baseline)": S.m.values,
                    "LinearTrend (baseline)": S.m.values + S.d_m_1h.values * AF.HORIZONS_H[hname]}
            for nm, p in base.items(): PRED[(fs, hname, nm, sp)] = pd.Series(p, index=S.index)
        for nm, mdl in model_zoo().items():
            tr = D[D.split == "train"]; mdl.fit(tr[cols], tr[tgt]); FITTED[(fs, hname, nm)] = mdl
            for sp in ["val", "test"]:
                S = D[D.split == sp]; PRED[(fs, hname, nm, sp)] = pd.Series(S.m.values + mdl.predict(S[cols]), index=S.index)
        for (f_, h_, nm, sp), p in [(k, v) for k, v in PRED.items() if k[0] == fs and k[1] == hname and k[3] in ("val", "test")]:
            S = D[D.split == sp]
            for farm in ["ALL"] + sorted(GRIDS):
                M = S if farm == "ALL" else S[S.farm == farm]
                pp = p
                if farm != "ALL": pp = pd.Series(p.values, index=S.index)[S.farm == farm]
                rows.append(dict(feature_set=fs, horizon=hname, model=nm, split=sp, farm=farm, **reg_metrics(M[f"y_{hname}"].values, pp.values, M.m.values)))
    print("trained feature set:", fs)
EVAL = pd.DataFrame(rows)
EVAL.to_csv(f"{OUT}/evaluation_regression.csv", index=False)

# --- C. Moisture-reduction (drying) analysis -------------------------------------------------
red = pd.concat([AF.reduction_table(g[AF.MOISTURE], STEP).assign(farm=f) for f, g in GRIDS.items()])
red.to_csv(f"{OUT}/moisture_reduction_summary.csv", index=False)
hod = pd.concat([ (g[AF.MOISTURE].shift(4) - g[AF.MOISTURE]).where(~((g[AF.MOISTURE].diff() > AF.RISE_PP).astype(float).rolling(4).max() > 0)).groupby(g.index.hour).median().rename(f) for f, g in GRIDS.items()], axis=1)
hod.to_csv(f"{OUT}/drying_by_hour_of_day.csv")
print("\n=== C. Moisture reduction (positive = drying; %-points per hour) ===")
print(red[red.subset == "no_wetting_event"].round(4).to_string(index=False))
ax = hod.plot(figsize=(9, 3.5), title="Median 1-hour moisture change by hour of day (wetting windows excluded)"); ax.axhline(0, c="k", lw=.5); show("drying_by_hour")

# --- A. Soil-moisture analysis: sensor validity + relative state --------------------------------
tr = sensor[sensor.Timestamp <= TRAIN_END]; te = sensor[sensor.Timestamp > VAL_END]
b1, b0 = np.polyfit(tr.Raw_Capacitive_ADC, tr.Volumetric_Moisture_Pct, 1)
res_tr = tr.Volumetric_Moisture_Pct - (b0 + b1 * tr.Raw_Capacitive_ADC); res_te = te.Volumetric_Moisture_Pct - (b0 + b1 * te.Raw_Capacitive_ADC)
FAULT_LIMIT = float(res_tr.abs().quantile(0.999) * 1.5)
print("\n=== A. Soil-moisture analysis ===")
print(f"ADC->moisture linear consistency: train MAE {res_tr.abs().mean():.3f} pp, test MAE {res_te.abs().mean():.3f} pp, "
      f"fault limit {FAULT_LIMIT:.2f} pp, false-alarm rate on clean test data {(res_te.abs() > FAULT_LIMIT).mean()*100:.2f}%")
print("(ADC and moisture come from the SAME sensor, so this only checks sensor health; it is not a prediction task.)")
rel = {"p10": float(train_m.quantile(.1)), "p25": float(train_m.quantile(.25)), "p50": float(train_m.quantile(.5)), "p75": float(train_m.quantile(.75)), "p90": float(train_m.quantile(.9))}
print("Relative moisture states come from this sensor's own history percentiles (NOT agronomic dry/wet):", {k: round(v, 2) for k, v in rel.items()})

## 8. Evaluate (honestly)
For every horizon we ask: *does the best model beat "moisture stays the same"?* The improvement is tested with a
**day-block bootstrap** (days are correlated, so rows are not independent). Only models that win on validation **and** test, with
a 95 % interval above zero, are marked `useful`.


In [ ]:
# 8. Evaluate
def day_block_ci(err_base, err_model, keys, n=2000):
    d = pd.DataFrame({"diff": err_base - err_model, "k": keys}).groupby("k")["diff"].agg(["sum", "count"])
    rng = np.random.default_rng(SEED); idx = rng.integers(0, len(d), (n, len(d)))
    means = d["sum"].values[idx].sum(1) / d["count"].values[idx].sum(1)
    return float(d["sum"].sum() / d["count"].sum()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

REG_TABLE, SELECTED = [], {}
for fs in ["deployable", "extended"]:
    for hname in AF.HORIZONS_H:
        D = build(fs, hname)
        v = EVAL[(EVAL.feature_set == fs) & (EVAL.horizon == hname) & (EVAL.split == "val") & (EVAL.farm == "ALL")].set_index("model").MAE
        cand = v.drop(["Persistence (baseline)", "LinearTrend (baseline)"])
        best = cand.idxmin()
        for sp in ["val", "test"]:
            S = D[D.split == sp]; y = S[f"y_{hname}"].values
            e_b = np.abs(y - PRED[(fs, hname, "Persistence (baseline)", sp)].values); e_m = np.abs(y - PRED[(fs, hname, best, sp)].values)
            keys = S.farm + S.index.strftime("%Y-%m-%d")
            if sp == "val": val_gain = day_block_ci(e_b, e_m, keys.values)
            else: test_gain = day_block_ci(e_b, e_m, keys.values)
        useful = bool(val_gain[0] > 0 and test_gain[0] > 0 and test_gain[1] > 0)
        S = D[D.split == "test"]; ev = S[f"event_in_{hname}"].values > 0
        e_b = np.abs(S[f"y_{hname}"].values - PRED[(fs, hname, "Persistence (baseline)", "test")].values); e_m = np.abs(S[f"y_{hname}"].values - PRED[(fs, hname, best, "test")].values)
        te_row = EVAL[(EVAL.feature_set == fs) & (EVAL.horizon == hname) & (EVAL.split == "test") & (EVAL.farm == "ALL")].set_index("model")
        SELECTED[(fs, hname)] = dict(model=best, useful_vs_persistence=useful, test_mae_gain_pp=test_gain[0], gain_ci95=[test_gain[1], test_gain[2]], val_mae_gain_pp=val_gain[0])
        REG_TABLE.append(dict(feature_set=fs, horizon=hname, selected_model=best, test_MAE=te_row.loc[best, "MAE"], test_RMSE=te_row.loc[best, "RMSE"], test_R2=te_row.loc[best, "R2"],
                              persistence_MAE=te_row.loc["Persistence (baseline)", "MAE"], persistence_R2=te_row.loc["Persistence (baseline)", "R2"],
                              trend_MAE=te_row.loc["LinearTrend (baseline)", "MAE"],
                              skill_pct_vs_persistence=100 * test_gain[0] / te_row.loc['Persistence (baseline)', 'MAE'], MAE_gain_vs_persistence=test_gain[0], gain_CI95_low=test_gain[1], gain_CI95_high=test_gain[2],
                              test_MAE_no_wetting_event=float(e_m[~ev].mean()), persistence_MAE_no_wetting_event=float(e_b[~ev].mean()),
                              test_MAE_with_wetting_event=float(e_m[ev].mean()) if ev.any() else np.nan,
                              persistence_MAE_with_wetting_event=float(e_b[ev].mean()) if ev.any() else np.nan, n_test=len(S), n_test_with_event=int(ev.sum()),
                              verdict="USEFUL (beats persistence, CI>0)" if useful else "NOT USEFUL (does not reliably beat persistence)"))
REG = pd.DataFrame(REG_TABLE); REG.to_csv(f"{OUT}/regression_summary.csv", index=False)
pd.set_option("display.width", 250)
print("=== B. Moisture forecasting – test set (last 15 % of the timeline) ===")
print(REG[["feature_set", "horizon", "selected_model", "test_MAE", "persistence_MAE", "trend_MAE", "test_R2", "persistence_R2", "skill_pct_vs_persistence", "MAE_gain_vs_persistence", "gain_CI95_low", "gain_CI95_high", "verdict"]].round(4).to_string(index=False))
print("\nAll models, test set, both farms pooled:")
print(EVAL[(EVAL.split == "test") & (EVAL.farm == "ALL") & (EVAL.feature_set == "deployable")][["horizon", "model", "MAE", "RMSE", "R2", "R2_delta"]].round(4).to_string(index=False))
print("\nPer-farm test MAE (deployable features):")
print(EVAL[(EVAL.split == "test") & (EVAL.farm != "ALL") & (EVAL.feature_set == "deployable")].pivot_table(index=["horizon", "model"], columns="farm", values="MAE").round(4).to_string())

# Cross-farm transfer: train on Farm 1 only, test on Farm 2 test period
print("\nCross-farm transfer (train Farm_1 -> test Farm_2, deployable features):")
xf = []
for hname in AF.HORIZONS_H:
    D = build("deployable", hname); cols = AF.FEATURE_SETS["deployable"]
    a = D[(D.split == "train") & (D.farm == "Kolkata_Farm_1")]; b = D[(D.split == "test") & (D.farm == "Kolkata_Farm_2")]
    pers = mean_absolute_error(b[f"y_{hname}"], b.m)
    for nm, mdl in model_zoo().items():
        mdl.fit(a[cols], a[f"delta_{hname}"]); p = b.m + mdl.predict(b[cols])
        xf.append(dict(horizon=hname, model=nm, MAE=mean_absolute_error(b[f"y_{hname}"], p), persistence_MAE=pers))
XF = pd.DataFrame(xf); XF.to_csv(f"{OUT}/cross_farm_transfer.csv", index=False); print(XF.round(4).to_string(index=False))

# Plot predicted vs actual (test, 1 h and 24 h)
fig, ax = plt.subplots(2, 1, figsize=(12, 7))
for a_, hname in zip(ax, ["1h", "24h"]):
    D = build("deployable", hname); S = D[(D.split == "test") & (D.farm == "Kolkata_Farm_2")]; sel = SELECTED[("deployable", hname)]["model"]
    p = PRED[("deployable", hname, sel, "test")]; p = pd.Series(p.values, index=D[D.split == "test"].index)[(D[D.split == "test"].farm == "Kolkata_Farm_2").values]
    a_.plot(S.index + pd.Timedelta(hours=AF.HORIZONS_H[hname]), S[f"y_{hname}"], label="actual", lw=1); a_.plot(S.index + pd.Timedelta(hours=AF.HORIZONS_H[hname]), p.values, label=f"{sel} prediction", lw=1)
    a_.plot(S.index + pd.Timedelta(hours=AF.HORIZONS_H[hname]), S.m, label="persistence", lw=.7, alpha=.6); a_.set_title(f"Farm_2 test period – {hname} ahead"); a_.legend()
show("predicted_vs_actual_test")

# --- Classification ----------------------------------------------------------------------------
# (1) Negative result worth knowing: a "dry" label defined from this sensor's own history cannot be evaluated chronologically.
Dc = build("deployable", "6h")
print("\n=== Classification (1): 'dry-risk' = moisture below the TRAIN 15th percentile (%.2f%%) within 6 h ===" % DRY_THR)
print("positive windows per split:", Dc.groupby("split").dry_risk.sum().astype(int).to_dict(), "of", Dc.groupby("split").size().to_dict())
print("-> validation and test contain ZERO positives (April is much wetter than Jan-Feb). Metrics would be meaningless, so none are reported.\n"
      "   Lesson: 'dry' cannot be defined from the data alone. It needs crop-specific thresholds from an agronomic reference + your calibration.")

# (2) Wetting-event classifier: will a rise > 1 pp (rain/irrigation-like, cause unlabeled) occur in the next 6 h?  Labels exist in every split.
print("\n=== Classification (2): wetting event within next 6 h ===")
clfs = {"LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)),
        "RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=10, class_weight="balanced", n_jobs=-1, random_state=SEED),
        "GradientBoosting": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3, class_weight="balanced", random_state=SEED)}
def cmetrics(y, p): return dict(accuracy=accuracy_score(y, p), balanced_accuracy=balanced_accuracy_score(y, p), precision=precision_score(y, p, zero_division=0),
                                recall=recall_score(y, p, zero_division=0), f1=f1_score(y, p, zero_division=0))
crow, CM = [], {}
for fs in ["deployable", "extended"]:
    Dc = build(fs, "6h"); cols = AF.FEATURE_SETS[fs]
    tr_, va_, te_ = (Dc[Dc.split == s_] for s_ in ["train", "val", "test"])
    ytr, yva, yte = (x[f"event_in_6h"].astype(int).values for x in (tr_, va_, te_))
    print(f"[{fs}] positive windows  train {ytr.sum()}/{len(ytr)}  val {yva.sum()}/{len(yva)}  test {yte.sum()}/{len(yte)}  (only ~11 separate events in test -> fragile)")
    base = np.zeros_like(yte); crow.append(dict(feature_set=fs, model="Always 'no event' (baseline)", **cmetrics(yte, base))); CM[f"{fs}|baseline"] = confusion_matrix(yte, base, labels=[0, 1]).tolist()
    for nm, c in clfs.items():
        c.fit(tr_[cols], ytr); pv = c.predict_proba(va_[cols])[:, 1]
        ths = np.linspace(.05, .95, 19); bt = ths[int(np.argmax([balanced_accuracy_score(yva, pv >= t) for t in ths]))]   # threshold chosen on VALIDATION only
        pt = (c.predict_proba(te_[cols])[:, 1] >= bt).astype(int)
        crow.append(dict(feature_set=fs, model=f"{nm} (thr={bt:.2f})", **cmetrics(yte, pt))); CM[f"{fs}|{nm}"] = confusion_matrix(yte, pt, labels=[0, 1]).tolist()
CLS = pd.DataFrame(crow); CLS.to_csv(f"{OUT}/evaluation_classification.csv", index=False); json.dump(CM, open(f"{OUT}/confusion_matrices.json", "w"), indent=1)
print(CLS.round(3).to_string(index=False)); print("Confusion matrices [[TN, FP],[FN, TP]] (test):"); [print("  ", k, v) for k, v in CM.items()]

# Per-crop evaluation: impossible with this workbook
PER_CROP = {"status": "UNSUPPORTED", "reason": "EcoIrrigate_Sensor_Data has no crop column; no sheet links soil-moisture readings to any crop.",
            "action": "Log crop_name and growth_stage with every ESP32 reading (template columns exist) and re-run retrain.py per crop once >= ~8-12 weeks / >= 30 irrigation cycles exist per crop."}
json.dump(PER_CROP, open(f"{OUT}/evaluation_per_crop.json", "w"), indent=1); print("\nPer-crop evaluation:", PER_CROP["status"], "-", PER_CROP["reason"])

## 9. Save trained models (with a registry that records whether each one is allowed to influence the pump)
**Important:** these models were trained on a *different sensor, soil and city*. Even if they pass the test above, they are
stored with `deployment_status = "shadow_only"` – they may log predictions but may not drive the pump until they pass the same
test on **your own ESP32 data** (`retrain.py`).


In [ ]:
# 9. Save
registry = {"created": dt.datetime.now().isoformat(timespec="seconds"), "workbook_sha256": sha, "step_minutes": STEP,
            "train_period": [str(t0), str(TRAIN_END)], "val_end": str(VAL_END), "test_end": str(t1), "dry_threshold_relative_pct": DRY_THR,
            "unit_warning": "Trained on Kolkata 'Volumetric_Moisture_Pct'. Your capacitive sensor has a different scale -> retrain on own data.",
            "adc_consistency": {"intercept": float(b0), "slope": float(b1), "fault_limit_pp": FAULT_LIMIT}, "relative_percentiles": rel, "models": {}}
for (fs, hname), sel in SELECTED.items():
    mdl = FITTED[(fs, hname, sel["model"])]; D = build(fs, hname); S = D[D.split == "val"]
    q90 = float(np.quantile(np.abs(S[f"delta_{hname}"].values - mdl.predict(S[AF.FEATURE_SETS[fs]])), 0.90))
    path = f"models/moisture_{fs}_{hname}.joblib"; joblib.dump(mdl, path)
    registry["models"][f"{fs}_{hname}"] = dict(path=path, model_type=sel["model"], horizon_h=AF.HORIZONS_H[hname], features=AF.FEATURE_SETS[fs],
        predicts="delta moisture (pp) over the horizon", val_abs_error_q90_pp=q90, passed_kolkata_test=sel["useful_vs_persistence"],
        test_mae_gain_pp=sel["test_mae_gain_pp"], gain_ci95=sel["gain_ci95"],
        deployment_status="shadow_only", may_control_pump=False,
        reason="not validated on the deployed sensor" if sel["useful_vs_persistence"] else "did not reliably beat persistence even on Kolkata test data")
json.dump(registry, open("models/model_registry.json", "w"), indent=1)
print(json.dumps({k: {x: v[x] for x in ("model_type", "passed_kolkata_test", "deployment_status")} for k, v in registry["models"].items()}, indent=1))
print("\nSaved: models/*.joblib, models/model_registry.json, crop_config.json, outputs/*.csv, outputs/plots/*.png")
print("Original workbook unchanged. SHA-256 now:", hashlib.sha256(open(WORKBOOK, "rb").read()).hexdigest() == sha)